# Spike-field coherence, Boran et al. (2019), dandiset 000574

Second notebook of the entry: the claim `sfc-maintenance-over-encoding`, kept apart because
it streams the microwire LFP (32 kHz). Dataset version 0.250815.1108 (see the first notebook's
header for why). It repeats the selection of hippocampal maintenance neurons of the first
notebook (paired permutation t test, maintenance against fixation, correct trials, atlas
regions) with the same generator.

In [ ]:
import time
import numpy as np
import pandas as pd
import sternberg as S
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
A = S.assets()
D = S.load_all(A, workers=16)

In [ ]:
units = []
for d in D:
    tr = d["trials"]
    lab = dict(zip(d["electrodes"].label, d["electrodes"].location))
    for u in d["units"]:
        loc1 = lab.get(u["bundle"] + "1", "?").split(",")[0]
        al = S.aligned_spikes(u["spikes"], tr.probe.to_numpy())
        units.append({"d": d, "u": u, "aligned": al, "correct": tr.correct.to_numpy(),
                      "atlas": {"Hipp": "hippocampus", "PhG": "entorhinal", "Amyg": "amygdala"}.get(loc1, "other")})
print(len(D), "sessions,", len(units), "units,", sum(x["atlas"] == "hippocampus" for x in units), "hippocampal (atlas)")
print("LFP shapes (samples x wires) and rates:", sorted({(d["shapes"]["lfp"][1], d["shapes"]["lfp"][0] / (8 * len(d["trials"]))) for d in D}))

## Selection

Hippocampal maintenance neurons (rate in -3 to 0 s above -6 to -5 s, paired permutation t test,
2000 sign flips, P < 0.05, correct trials), with at least 100 spikes in the last 2 s of
maintenance and in encoding (fig. S6b legend: N = 94). The paper's "at least 100 spikes
during both maintenance and encoding" is read on the windows the coherence uses.

In [ ]:
rng = np.random.default_rng(701)
sel = []
for x in units:
    if x["atlas"] != "hippocampus":
        continue
    c = x["correct"]
    m = S.count_in(x["aligned"], S.MAINT)[c] / 3.0
    f = S.count_in(x["aligned"], S.FIX)[c] / 1.0
    d_ = m - f
    t_ref = S.t_paired_rows(rng.choice([-1.0, 1.0], (2000, len(d_))) * d_)
    t0 = S.t_paired_rows(d_[None])[0]
    if S.perm_p(t0, t_ref) < 0.05 and t0 > 0:
        nm = S.count_in(x["aligned"], S.LAST2)[c].sum()
        ne = S.count_in(x["aligned"], S.ENC)[c].sum()
        if nm >= 100 and ne >= 100:
            sel.append(x)
wires = sorted({(x["d"]["asset_id"], x["u"]["electrode_row"], x["u"]["bundle"]) for x in sel})
print(f"selected neurons: {len(sel)} (paper 94), on {len(wires)} microwires")

## Streaming the LFP

Each wire's LFP column is read whole and decimated to 1 kHz; the bundle's bipolar iEEG (contact
1 minus 2, 1 kHz) and one scalp EEG channel (Pz where recorded, otherwise O1, the most common
maximal-PLV electrode of table S1; an assumption, the paper does not say which) are read with it.

In [ ]:
def scalp_for(asset_id):
    d = [d for d in D if d["asset_id"] == asset_id][0]
    labs = list(d["electrodes"][d["electrodes"].group_name == "eeg"].label)
    return "Pz" if "Pz" in labs else "O1"


t0 = time.time()
W = S.load_wires([(a, r, b, scalp_for(a)) for a, r, b in wires], workers=20)
W = dict(zip(wires, W))
print(f"read {len(W)} wires in {time.time() - t0:.0f} s")

## Pairwise phase consistency

PPC (P-hat-2, Vinck et al. 2012, FieldTrip's ppc2) between each neuron's spikes and the LFP of
its own wire, per frequency (1 Hz steps), in the last 2 s of maintenance and in encoding (-5 to
-3 s), correct trials. Phase at each spike from a Hanning-tapered window of 3 or 5 cycles
(readings; the paper gives no window). Significance per bin against the 95th percentile of 200
PPCs with the spike trains assigned to permuted trials (Methods, Spike-field coherence).
Frequency ranges (a) 4 to 30 Hz, (b) 1 to 40 Hz. The same for the bundle's iEEG and for scalp
EEG.

In [ ]:
rng = np.random.default_rng(702)
FREQS = np.arange(1, 41)


def ppc_sig(x, sig_name, cycles, fs):
    key = (x["d"]["asset_id"], x["u"]["electrode_row"], x["u"]["bundle"])
    if sig_name not in W[key]:
        return None
    sig = W[key][sig_name]
    c = np.flatnonzero(x["correct"])
    Zc = S.spike_phases_setup(sig[c].astype(np.float64), fs, FREQS, cycles)
    res = {}
    for per, (lo, hi) in [("maintenance", S.LAST2), ("encoding", S.ENC)]:
        idx, tri = [], []
        for j, k in enumerate(c):
            s = x["aligned"][k]
            s = s[(s >= lo) & (s < hi)]
            idx.append(np.round((s + 6.0) * fs).astype(int))
            tri.append(np.full(len(s), j))
        idx, tri = np.concatenate(idx), np.concatenate(tri)
        idx = np.clip(idx, 0, sig.shape[1] - 1)
        obs = S.ppc2(Zc[tri, idx], tri)
        null = np.array([S.ppc2(Zc[p[tri], idx], tri) for p in (rng.permutation(len(c)) for _ in range(200))])
        res[per] = obs > np.nanpercentile(null, 95, axis=0)
    return res


rows = {}
for sig_name, fs in [("lfp", S.SFC_FS), ("ieeg", S.SFC_FS), ("eeg", S.EEG_FS)]:
    for cycles in (3, 5):
        R = [ppc_sig(x, sig_name, cycles, fs) for x in sel]
        R = [r for r in R if r is not None]
        M = np.array([r["maintenance"] for r in R])
        E = np.array([r["encoding"] for r in R])
        rows[(sig_name, cycles)] = (100 * M.mean(0), 100 * E.mean(0), len(R))
for (sig_name, cycles), (m, e, n) in rows.items():
    for fr in [(4, 30), (1, 40)]:
        k = (FREQS >= fr[0]) & (FREQS <= fr[1])
        print(f"{sig_name}, {cycles} cycles, {fr[0]}-{fr[1]} Hz, {n} neurons: maintenance % per bin {np.round(m[k], 1).tolist()}")
        print(f"   encoding % per bin {np.round(e[k], 1).tolist()}")
        print(f"   bins with maintenance above encoding: {int(np.sum(m[k] > e[k]))} of {int(k.sum())}; "
              f"mean % maintenance {m[k].mean():.1f}, encoding {e[k].mean():.1f}; max maintenance {m[k].max():.1f}")

**Result: Partially reproduced.** 73 hippocampal maintenance neurons have at least 100 spikes in the last 2 s of maintenance and in encoding (paper 94 +/- 9.4, outside), on 47 microwires. With 3-cycle windows over 4 to 30 Hz, the percentage of neurons with significant PPC to the LFP of their own wire is higher in maintenance than in encoding in all 27 bins (means 58.4% against 42.1%), and in 39 of 40 bins over 1 to 40 Hz; with 5-cycle windows in 26 of 27 and 38 of 40. Coupling to the bundle's iEEG is also well above the 5% chance level (maintenance means 29.6% and 21.0% of neurons over 4 to 30 Hz with 3 and 5 cycles), and to the scalp EEG it is near chance on average (5.3% and 5.0%) with single bins up to 16.4%, so the part of the claim that the coupling is not to the iEEG or scalp EEG does not hold.

## Summary

| Claim | Outcome |
|---|---|
| sfc-maintenance-over-encoding | Partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method: the window around each spike (3 or 5 cycles, Hanning) is not given by the paper; the LFP is decimated to 1 kHz; the scalp EEG channel is Pz or, where Pz was not recorded, O1 (assumption); 73 neurons where the paper has 94. What was found about the dataset: the microwire LFP is present for all sessions, but in subject 2 session 5 it holds 11,264,000 samples for 46 trials, not 32 kHz x 8 s per trial (its rate per stored second is 30,609 Hz in the sanity output); no wire of that session entered this analysis.